# LEP-i実関数の検証：任意の積分下限での真値からの誤差

**設定セルの `ENERGY_MIN_EVQ` で積分下限を指定し、公開関数の密度・速度・温度・圧力を全速度空間の既知の真値と比較する。** 上限は `ENERGY_MAX_EV`（既定値30 keV/q）。30 eV/qとの差は補足として示す。

対象はH⁺の定常・等方的なドリフトMaxwell分布。LEP-i L2形式の疑似fluxをインストール済みの公開関数に入力し、独立な離散計算とも照合する。mock・monkeypatch・実観測flux・ネットワーク取得は使わない。

- 下限30 eV/qの基礎検証：`LEPi_part_products_synthetic_validation_Emin30eVq.ipynb`。
- 連続積分の基準：`LEPi_drifting_Maxwellian_partial_moments.ipynb`。
- このノート：指定下限での真値からの誤差、続いて30 eV/qとの差。

H⁺ではeVとeV/qの数値は一致する。公開関数はビン中心値を下限以上・上限以下で選ぶ。ビン中心に一致する下限では、そのビンを含む。同じビンが選ばれる下限間で公開関数の出力は変わらないが、指定範囲の連続積分は変わる。

`ENERGY_MIN_EVQ` を変更したら全セルを上から実行する。CSV・図・実装情報は `outputs/Emin{設定下限}eVq/` に保存する。同じ下限で再実行するとその保存先を更新する。ファイル名の `Emin100eVq` は従来の名称として維持しており、現在の下限は設定値と実行結果で確認する。


## 誤差の定義と仮定

真値は密度 $n$、速度 $\mathbf{U}$、温度エネルギー $\theta=k_{\mathrm{B}}T$、圧力 $p=n\theta$。スカラー温度・圧力は、それぞれの積分下限で得たpartial平均速度を中心に計算する。

主結果は、全分布の真値 $Q_{\mathrm{true}}$ に対する符号付き相対誤差

$$
\varepsilon_Q=100\left(\frac{Q_{\mathrm{target}}}{Q_{\mathrm{true}}}-1\right)\quad[\%]
$$

で評価する。速度はドリフト方向への射影に対する誤差と、速度差（km/s）を記録する。$U=0$ では相対速度誤差を定義せず、絶対速度差を使う。全分布に対する誤差には、未積分領域による偏りと、有限ビンでの処理の影響が両方含まれる。独立な離散計算との不一致とは別の量である。

補足の下限間の出力差は、参照下限30 eV/qの $Q_{\mathrm{ref}}$、指定下限の $Q_{\mathrm{target}}$ に対して

$$
\delta_Q=100\left(\frac{Q_{\mathrm{target}}}{Q_{\mathrm{ref}}}-1\right)\quad[\%]
$$

とする。速度は入力ドリフト方向への射影を使い、$U=0$ の速度比は定義しない。速度差 $U_{\mathrm{target}}-U_{\mathrm{ref}}$（km/s）も記録する。各出力と真値の比も併記し、「両下限で近い」と「真値に近い」を区別する。

$x$、$y$、$z$ driftはそれぞれ $\mathbf{U}=(U,0,0),(0,U,0),(0,0,U)$。getterの角度を表すCartesian軸であり、FAC軸ではない。分布は定常・等方的な単一Maxwell分布。中心サンプルを主比較に使い、1 keVでは一様top-hatビン平均の感度も確認する。検出器の校正済み応答、衛星電位、未観測の冷たい成分、非Maxwell分布、He⁺・O⁺は含めない。

流速・温度範囲は例示であり、実際のtime rangeの分布を拘束するものではない。密度回収率 $\le10^{-10}$ の温度・圧力・速度の比は解釈しない。圧力の絶対値には $n=1\,\mathrm{cm}^{-3}$ を使うが、比と出力差はこの値に依存しない。


In [ ]:
from pathlib import Path
import hashlib
import json
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import lepi_synthetic_validation as validation

PRIMARY_PACKAGE = 'pyspedas'
ENERGY_MAX_EV = 30_000.0
# H+ではeV/qとeVの数値が一致する。積分下限はこの一か所で指定する。
ENERGY_MIN_EVQ = 1000.0
REFERENCE_ENERGY_MIN_EVQ = 30.0
if not (np.isfinite(ENERGY_MIN_EVQ) and 0 <= ENERGY_MIN_EVQ < ENERGY_MAX_EV):
    raise ValueError('ENERGY_MIN_EVQ must be finite and satisfy 0 <= Emin < Emax.')
ENERGY_MINIMA_EV = list(dict.fromkeys([REFERENCE_ENERGY_MIN_EVQ, ENERGY_MIN_EVQ]))
OUTPUT_TAG = f'Emin{ENERGY_MIN_EVQ:g}eVq'
OUTPUT_DIR = Path.cwd() / 'outputs' / OUTPUT_TAG
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REFERENCE_DENSITY_CM3 = 1.0
TEMPERATURES_EV = np.array([1., 3., 10., 30., 100., 300., 1000., 3000., 10000.])
SPEEDS_KMS = np.array([0., 10., 30., 50., 100., 200., 300.])
DIRECTIONS = {'x': np.array([1.,0.,0.]), 'y': np.array([0.,1.,0.]), 'z': np.array([0.,0.,1.])}
plt.rcParams.update({'font.size':11, 'figure.dpi':110})
manifest = validation.source_manifest()
manifest.update(python=sys.version, energy_min_evq=ENERGY_MIN_EVQ, reference_energy_min_evq=REFERENCE_ENERGY_MIN_EVQ, energy_minima_ev=ENERGY_MINIMA_EV, energy_max_ev=ENERGY_MAX_EV,
                temperatures_ev=TEMPERATURES_EV.tolist(), speeds_kms=SPEEDS_KMS.tolist())
for name in ['lepi_synthetic_validation.py', 'lepi_l2_geometry.json',
             'LEPi_drifting_Maxwellian_partial_moments.ipynb']:
    manifest[name+'_sha256'] = hashlib.sha256(Path(name).read_bytes()).hexdigest()
(OUTPUT_DIR/'manifest.json').write_text(json.dumps(manifest, indent=2, ensure_ascii=False)+'\n')
display(pd.DataFrame([manifest['packages']]))
print('Output directory:', OUTPUT_DIR)


## 実際に選ばれるビン

下限を変えても、境界をまたぐビンを部分的に積分する処理にはならない。中心値による採否と、その幅に対応する境界を示す。後の比較では、要求した球殻の連続積分に加え、選択ビンの外側境界での連続基準も記録する。


In [ ]:
bin_rows, support_rows = [], []
for emin in ENERGY_MINIMA_EV:
    table = pd.DataFrame(validation.native_bin_table(PRIMARY_PACKAGE, [emin, ENERGY_MAX_EV]))
    table['energy_min_ev'] = emin
    bin_rows.append(table)
    if not table.selected.any():
        raise ValueError(f'No native energy bins selected for Emin={emin:g} eV/q.')
    selected = table[table.selected]
    support_rows.append(dict(energy_min_ev=emin, n_selected=len(selected),
                             minimum_centre_ev=selected.energy_ev.min(),
                             maximum_centre_ev=selected.energy_ev.max(),
                             lower_support_ev=selected.getter_lower_edge_ev.min(),
                             upper_support_ev=selected.getter_upper_edge_ev.max()))
bins = pd.concat(bin_rows, ignore_index=True)
supports = pd.DataFrame(support_rows)
bins.to_csv(OUTPUT_DIR/'selected_bins.csv', index=False)
supports.to_csv(OUTPUT_DIR/'selected_supports.csv', index=False)
display(supports)
fig, axes = plt.subplots(len(ENERGY_MINIMA_EV), 1, figsize=(10, 2*len(ENERGY_MINIMA_EV)), constrained_layout=True, squeeze=False)
for ax, emin in zip(axes[:, 0], ENERGY_MINIMA_EV):
    selected = bins[(bins.energy_min_ev==emin) & bins.selected]
    ax.errorbar(selected.energy_ev, np.zeros(len(selected)),
                xerr=[selected.energy_ev-selected.getter_lower_edge_ev,
                      selected.getter_upper_edge_ev-selected.energy_ev], fmt='o', markersize=4)
    ax.axvline(emin, color='0.4', ls='--')
    ax.axvline(ENERGY_MAX_EV, color='0.4', ls='--')
    ax.set_xscale('log')
    ax.set_xlim(min(20., max(min(ENERGY_MINIMA_EV)*0.8, 0.01)), ENERGY_MAX_EV*1.3)
    ax.set_yticks([])
    ax.set_title(rf'$E_{{\min}}={emin:g}\,\mathrm{{eV}}$')
    ax.set_xlabel(r'Energy $E$ [$\mathrm{eV}$]')
fig.savefig(OUTPUT_DIR/'selected_bins.png', dpi=180)
plt.show()


## 実公開関数の走査と独立計算との照合

両下限のすべての点で、密度・ベクトルflux・速度・圧力テンソル・温度テンソル・平均温度を、独立な離散計算と照合する。疑似flux生成は積分下限を引数に取らず、同じ生成条件で同じ入力配列になる。連続基準にもそれぞれの下限を適用する。


In [ ]:
rows = []
for direction, nhat in DIRECTIONS.items():
    for theta in TEMPERATURES_EV:
        for speed in SPEEDS_KMS:
            for emin in ENERGY_MINIMA_EV:
                r = validation.result_record(theta, speed*nhat, REFERENCE_DENSITY_CM3,
                                             PRIMARY_PACKAGE, energy_range=[emin, ENERGY_MAX_EV])
                rows.append(dict(direction=direction, **r))
    print('Completed direction:', direction)
scan = pd.DataFrame(rows)
scan.to_csv(OUTPUT_DIR/'public_parameter_scan.csv', index=False)
print('Actual public-function calls:', len(scan))
print('Maximum scaled difference from independent discrete calculation:', scan.max_discrete_scaled_error.max())
print('Runtime/logging warnings:', scan.public_warning_count.sum(), scan.public_logging_warning_count.sum())

# If earlier generated results are available, verify unchanged 30 eV behaviour.
old_results = Path('outputs/public_parameter_scan.csv')
if old_results.exists():
    old = pd.read_csv(old_results)
    check = scan[scan.energy_min_ev==30.].merge(old, on=['theta_ev','true_speed_kms','direction'],
                                              suffixes=('_new','_old'), validate='one_to_one')
    assert len(check) == 189
    for field in ['density_ratio','projected_speed_kms','temperature_ratio','pressure_ratio']:
        np.testing.assert_allclose(check[field+'_new'], check[field+'_old'], rtol=1e-12, atol=1e-10, equal_nan=True)
    print('30 eV results agree with the previous notebook outputs.')


## 主結果：指定下限での真値からの誤差

ここでの真値は**全速度空間の分布**の密度・バルク速度・温度・圧力である。指定下限以上の連続積分を真値とはしない。その連続partial momentも図に重ね、範囲制限だけで生じる偏りと、実関数のネイティブビンを含む誤差を読み分ける。

主比較はビン中心でサンプルしたfluxを使う。ビン内分布の扱いを変えた感度は後半に示す。


In [ ]:
error_target = scan[scan.energy_min_ev==ENERGY_MIN_EVQ].copy()
for name in ['density','velocity','temperature','pressure']:
    error_target[name+'_error_pct'] = 100*(error_target[name+'_ratio']-1)
error_target['velocity_error_kms'] = error_target.projected_speed_kms-error_target.true_speed_kms
error_target['temperature_error_ev'] = error_target.temperature_ev-error_target.theta_ev
error_target.to_csv(OUTPUT_DIR/f'{OUTPUT_TAG}_errors_vs_truth.csv',index=False)

one_kev_target = error_target[error_target.theta_ev==1000.].copy()
error_fields = ['density_error_pct','velocity_error_pct','temperature_error_pct','pressure_error_pct']
truth_error_summary = one_kev_target[error_fields].agg(['min','max'])
truth_error_summary.to_csv(OUTPUT_DIR/f'one_kev_{OUTPUT_TAG}_error_ranges.csv')
display(truth_error_summary)
display(one_kev_target[one_kev_target.true_speed_kms==100.][
    ['direction','density_ratio','projected_speed_kms','velocity_error_pct',
     'temperature_ev','temperature_error_pct','pressure_ratio','pressure_error_pct']])

fig,axes=plt.subplots(1,4,figsize=(18,4.5),constrained_layout=True)
for ax,name,title in zip(axes,['density','velocity','temperature','pressure'],
                         ['Density','Velocity','Temperature','Pressure']):
    for direction in DIRECTIONS:
        sub=one_kev_target[one_kev_target.direction==direction]
        ax.plot(sub.true_speed_kms,sub[name+'_error_pct'],'o-',label=direction+' drift: public')
    ref=one_kev_target[one_kev_target.direction=='x']
    if name=='velocity':
        continuous_error=np.where(ref.true_speed_kms>0,
                      100*(ref.continuous_speed_kms/ref.true_speed_kms-1),np.nan)
    else:
        continuous_error=100*(ref['continuous_'+name+'_ratio']-1)
    ax.plot(ref.true_speed_kms,continuous_error,'k:',label='Continuous partial moment')
    ax.axhline(0.,color='0.6',ls='--')
    ax.set_xlabel(r'True drift speed $U$ [$\mathrm{km\,s^{-1}}$]')
    ax.set_ylabel('Signed error relative to full-distribution truth [%]')
    ax.set_title(title)
    ax.grid(alpha=.3)
    ax.legend(fontsize=8)
fig.suptitle(rf'H$^{{+}}$ Maxwellian: $\theta=1\,\mathrm{{keV}}$, $E_{{\min}}={ENERGY_MIN_EVQ:g}\,\mathrm{{eV/q}}$')
fig.savefig(OUTPUT_DIR/f'one_kev_{OUTPUT_TAG}_errors_vs_truth.png',dpi=180)
plt.show()


## 主結果：真値に対する誤差の数値表

指定下限の全走査点について、$100(Q_{\mathrm{out}}/Q_{\mathrm{true}}-1)$ を温度・drift方向ごとにprintする。小さい値も読み取れるよう、有効数字6桁（必要なら科学表記）で表示する。密度・速度・温度・圧力の誤差は%、速度差は入力drift方向への射影でkm/s、温度差はeV。

$U=0$ の相対速度誤差は未定義。密度回収率が閾値以下の点では、速度・温度・圧力の誤差および速度差・温度差を解釈しない。これらは `NaN` と表示する。数値表を `errors_vs_truth_print_table.csv` に保存する。

In [ ]:
print(f'Full-distribution truth errors: Emin={ENERGY_MIN_EVQ:g} eV/q, '
      f'Emax={ENERGY_MAX_EV:g} eV/q')
print('Signed error [%] = 100 * (output / truth - 1); velocity uses drift-direction projection.')
print('NaN: undefined U=0 relative velocity error, or insufficient density recovery.')
print('Values: 6 significant digits; CSV retains full numerical precision.\n')

print_columns = ['true_speed_kms', 'density_ratio', 'density_error_pct',
                 'velocity_error_pct', 'temperature_error_pct', 'pressure_error_pct',
                 'velocity_error_kms', 'temperature_error_ev']
error_print_table = error_target[['theta_ev', 'direction'] + print_columns].copy()
invalid_density = error_print_table.density_ratio <= validation.DENSITY_FLOOR
error_print_table.loc[invalid_density, [
    'velocity_error_pct', 'temperature_error_pct', 'pressure_error_pct',
    'velocity_error_kms', 'temperature_error_ev']] = np.nan
error_print_table.loc[error_print_table.true_speed_kms == 0, 'velocity_error_pct'] = np.nan
error_print_table.to_csv(OUTPUT_DIR/'errors_vs_truth_print_table.csv', index=False)

column_labels = {'true_speed_kms': 'U [km/s]', 'density_ratio': 'n_out/n',
                 'density_error_pct': 'dn [%]', 'velocity_error_pct': 'dU [%]',
                 'temperature_error_pct': 'dtheta [%]', 'pressure_error_pct': 'dp [%]',
                 'velocity_error_kms': 'dU [km/s]', 'temperature_error_ev': 'dtheta [eV]'}
for theta in sorted(error_print_table.theta_ev.unique()):
    for direction in DIRECTIONS:
        table = error_print_table[(error_print_table.theta_ev == theta)
                                  & (error_print_table.direction == direction)]
        print(f'True theta = {theta:g} eV; {direction} drift')
        print(table.sort_values('true_speed_kms')[print_columns].rename(columns=column_labels)
              .to_string(index=False, float_format=lambda value: f'{value:.6g}', na_rep='NaN'))
        print()


## 主結果：真値比の温度・流速依存性（30 eV/qの図と同形式）

30 eV/qの検証図と同じく、密度・速度・温度・圧力の **出力／全分布の真値** を、$x,y,z$ driftの3行・4量の4列で表示する。比1が真値との一致に対応する。

配色、軸、添字、マスク、図サイズを揃え、各量のカラースケールは参照下限・指定下限の全ケースを合わせて決める。指定下限の図に続けて、同じ共通スケールの30 eV/q図も併記する。元の30 eV/qノートの自動スケールとは範囲が異なる場合がある。

流速ゼロの速度比と、密度回収率の閾値により温度・速度・圧力の比を解釈しない点は灰色にする。符号付き誤差（%）の表と1 keVの誤差曲線は前のセルに残す。

In [ ]:
fields = [('density_ratio', r'$n_{\mathrm{out}}/n$'),
          ('velocity_ratio', r'$(\mathbf{U}_{\mathrm{out}}\cdot\hat{\mathbf{U}})/U$'),
          ('temperature_ratio', r'$\theta_{\mathrm{out}}/\theta$'),
          ('pressure_ratio', r'$p_{\mathrm{out}}/p$')]

# Compute shared limits from the values actually shown at both cutoffs.
shared_limits = {}
for field, _ in fields:
    values = scan[field].to_numpy().copy()
    if field != 'density_ratio':
        values[scan.density_ratio.to_numpy() <= validation.DENSITY_FLOOR] = np.nan
    finite = values[np.isfinite(values)]
    shared_limits[field] = (finite.min(), finite.max())

display(pd.DataFrame(shared_limits, index=['vmin', 'vmax']))

for emin in dict.fromkeys([ENERGY_MIN_EVQ, REFERENCE_ENERGY_MIN_EVQ]):
    selected_scan = scan[scan.energy_min_ev == emin]
    fig, axes = plt.subplots(3, 4, figsize=(18, 12), constrained_layout=True)
    for row, direction in enumerate(DIRECTIONS):
        subset = selected_scan[selected_scan.direction == direction]
        for col, (field, title) in enumerate(fields):
            ax = axes[row, col]
            grid = subset.pivot(index='theta_ev', columns='true_speed_kms', values=field)
            values = grid.to_numpy().copy()
            if field != 'density_ratio':
                density_grid = subset.pivot(index='theta_ev', columns='true_speed_kms', values='density_ratio')
                values[density_grid.to_numpy() <= validation.DENSITY_FLOOR] = np.nan
            vmin, vmax = shared_limits[field]
            im = ax.pcolormesh(grid.columns, grid.index, values, shading='nearest',
                               cmap=plt.get_cmap('viridis').with_extremes(bad='0.8'),
                               vmin=vmin, vmax=vmax)
            ax.set_yscale('log')
            ax.set_xlabel(r'True $U$ [$\mathrm{km\,s^{-1}}$]')
            ax.set_ylabel(r'True $\theta$ [$\mathrm{eV}$]')
            ax.set_title(direction+' drift: '+title)
            fig.colorbar(im, ax=ax)
    fig.suptitle(PRIMARY_PACKAGE+': actual public function on native LEP-i bins; '
                 +rf'$E_{{\min}}={emin:g}\,\mathrm{{eV}}$')
    fig.savefig(OUTPUT_DIR/f'public_parameter_scan_Emin{emin:g}_shared_scale.png', dpi=160)
    plt.show()


In [ ]:
def pair_cutoffs(frame):
    keys = ['theta_ev','true_speed_kms','direction','sampling','package']
    pair = frame[frame.energy_min_ev==REFERENCE_ENERGY_MIN_EVQ].merge(frame[frame.energy_min_ev==ENERGY_MIN_EVQ],
                   on=keys, suffixes=('_ref','_target'), validate='one_to_one')
    valid = (pair.density_ratio_ref > validation.DENSITY_FLOOR) & (pair.density_ratio_target > validation.DENSITY_FLOOR)
    fields = {'density':'density_ratio', 'velocity':'projected_speed_kms',
              'temperature':'temperature_ratio', 'pressure':'pressure_ratio'}
    for label, field in fields.items():
        denominator = pair[field+'_ref']
        allowed = valid & np.isfinite(denominator) & (np.abs(denominator)>1e-12)
        if label=='velocity': allowed &= pair.true_speed_kms>0
        pair[label+'_change_pct'] = np.where(allowed, 100*(pair[field+'_target']/denominator-1), np.nan)
    pair['velocity_change_kms'] = np.where(valid, pair.projected_speed_kms_target-pair.projected_speed_kms_ref, np.nan)
    for label in ['density','temperature','pressure']:
        prefix = 'continuous_'+label+'_ratio'
        allowed = valid & (np.abs(pair[prefix+'_ref'])>1e-12)
        pair['continuous_'+label+'_change_pct'] = np.where(allowed,100*(pair[prefix+'_target']/pair[prefix+'_ref']-1),np.nan)
    allowed = valid & (pair.true_speed_kms>0) & (np.abs(pair.continuous_speed_kms_ref)>1e-12)
    pair['continuous_velocity_change_pct'] = np.where(allowed,100*(pair.continuous_speed_kms_target/pair.continuous_speed_kms_ref-1),np.nan)
    return pair

paired = pair_cutoffs(scan)
paired.to_csv(OUTPUT_DIR/'paired_cutoff_comparison.csv', index=False)


## 補足：1 keVでの30 eV/qとの比較

比と差を分けて図示する。$Q_{\mathrm{target}}/Q_{\mathrm{ref}}$ が1に近いかと、$Q_{\mathrm{target}}/Q_{\mathrm{true}}$ が1に近いかは別の判断である。公開関数のネイティブビンでの結果と、理想的な球殻の連続基準も並べる。


In [ ]:
one_kev = paired[paired.theta_ev==1000.].copy()
change_fields = ['density_change_pct','velocity_change_pct','temperature_change_pct','pressure_change_pct']
summary = one_kev[change_fields].agg(['min','max'])
summary.to_csv(OUTPUT_DIR/'one_kev_change_ranges.csv')
display(summary)
display(one_kev[one_kev.true_speed_kms==100.][['direction','projected_speed_kms_ref','projected_speed_kms_target',
                       'temperature_ev_ref','temperature_ev_target','pressure_ratio_ref','pressure_ratio_target']])

fig, axes = plt.subplots(2, 4, figsize=(18, 8), constrained_layout=True)
fields = [('density_ratio',r'$n_{\mathrm{out}}/n$','density'),
          ('velocity_ratio',r'$U_{\mathrm{out}}/U$','velocity'),
          ('temperature_ratio',r'$\theta_{\mathrm{out}}/\theta$','temperature'),
          ('pressure_ratio',r'$p_{\mathrm{out}}/p$','pressure')]
colors = {'x':'C0','y':'C1','z':'C2'}
for col,(field,title,label) in enumerate(fields):
    for direction in DIRECTIONS:
        sub = one_kev[one_kev.direction==direction]
        for emin, style, key in [(REFERENCE_ENERGY_MIN_EVQ, '-', 'ref'), (ENERGY_MIN_EVQ, '--', 'target')]:
            axes[0,col].plot(sub.true_speed_kms,sub[field+'_'+key],style,
                             color=colors[direction],marker='o',markersize=3,
                             label=rf'{direction}: ${emin:g}\,\mathrm{{eV}}$')
        axes[1,col].plot(sub.true_speed_kms,sub[label+'_change_pct'],'o-',
                         color=colors[direction],label=direction+' drift')
    ref = one_kev[one_kev.direction=='x']
    axes[1,col].plot(ref.true_speed_kms,ref['continuous_'+label+'_change_pct'],'k:',label='Continuous shell')
    axes[0,col].axhline(1.,color='0.6',ls=':')
    axes[0,col].set_ylabel(title)
    axes[1,col].axhline(0.,color='0.6',ls=':')
    axes[1,col].set_ylabel(r'$100(Q_{\mathrm{target}}/Q_{\mathrm{ref}}-1)$ [%]')
    for row in [0,1]:
        axes[row,col].set_xlabel(r'True drift speed $U$ [$\mathrm{km\,s^{-1}}$]')
        axes[row,col].grid(alpha=.3)
        axes[row,col].legend(fontsize=8)
fig.suptitle(r'Single H$^{+}$ Maxwellian: $\theta=1\,\mathrm{keV}$')
fig.savefig(OUTPUT_DIR/'one_kev_cutoff_comparison.png',dpi=180)
plt.show()


## 補足：温度・流速依存性

下限を参照値から指定値に変更した際の出力変化（%）を全温度範囲で表示する。流速ゼロの速度比、および密度回収率が閾値以下の比較は灰色にする。「だいたい同じ」と呼ぶ許容差は研究の要求に依存するため、このノートでは恣意的な合否閾値を付けない。


In [ ]:
fig,axes = plt.subplots(3,4,figsize=(18,12),constrained_layout=True)
for row,direction in enumerate(DIRECTIONS):
    sub=paired[paired.direction==direction]
    for col,(field,title) in enumerate(zip(change_fields,['Density','Velocity','Temperature','Pressure'])):
        grid=sub.pivot(index='theta_ev',columns='true_speed_kms',values=field)
        all_values=paired[field].to_numpy()
        limit=np.nanmax(np.abs(all_values))
        cmap=plt.get_cmap('coolwarm').with_extremes(bad='0.8')
        ax=axes[row,col]
        im=ax.pcolormesh(grid.columns,grid.index,grid.to_numpy(),shading='nearest',
                         cmap=cmap,vmin=-limit,vmax=limit)
        ax.set_yscale('log')
        ax.set_xlabel(r'True $U$ [$\mathrm{km\,s^{-1}}$]')
        ax.set_ylabel(r'True $\theta$ [$\mathrm{eV}$]')
        ax.set_title(direction+' drift: '+title+' change [%]')
        fig.colorbar(im,ax=ax)
fig.suptitle(rf'Change from $E_{{\min}}={REFERENCE_ENERGY_MIN_EVQ:g}$ to ${ENERGY_MIN_EVQ:g}\,\mathrm{{eV/q}}$')
fig.savefig(OUTPUT_DIR/'cutoff_change_parameter_map.png',dpi=160)
plt.show()


## 補足：実装間の一致とビン平均への感度

代表ケースについて両下限で `pyspedas` / `ergpyspedas` の出力を比較する。1 keVでは中心サンプルと一様top-hatビン平均を比較する。後者も実際の校正応答ではない。


In [ ]:
implementation_rows=[]
for package in ['pyspedas','ergpyspedas']:
    for theta,speed,direction in [(30.,30.,'x'),(1000.,100.,'x'),(1000.,100.,'z'),(10000.,100.,'y')]:
        for emin in ENERGY_MINIMA_EV:
            r=validation.result_record(theta,speed*DIRECTIONS[direction],package=package,
                                       energy_range=[emin,ENERGY_MAX_EV])
            implementation_rows.append(dict(direction=direction,**r))
implementations=pd.DataFrame(implementation_rows)
keys=['theta_ev','true_speed_kms','direction','energy_min_ev']
check=implementations[implementations.package=='pyspedas'].merge(
    implementations[implementations.package=='ergpyspedas'],on=keys,suffixes=('_psp','_erg'),validate='one_to_one')
for field in ['density_ratio','projected_speed_kms','temperature_ratio','pressure_ratio']:
    np.testing.assert_allclose(check[field+'_psp'],check[field+'_erg'],rtol=1e-12,atol=1e-10)
implementations.to_csv(OUTPUT_DIR/'implementation_comparison.csv',index=False)
print('Both public implementations agree at both cutoffs for the tested cases.')

sampling_rows=[]
for direction,nhat in DIRECTIONS.items():
    for sampling in ['centre','bin_average']:
        for emin in ENERGY_MINIMA_EV:
            r=validation.result_record(1000.,100.*nhat,sampling=sampling,energy_range=[emin,ENERGY_MAX_EV])
            sampling_rows.append(dict(direction=direction,**r))
sampling_long=pd.DataFrame(sampling_rows)
sampling_pair=pair_cutoffs(sampling_long)
sampling_pair.to_csv(OUTPUT_DIR/'sampling_sensitivity.csv',index=False)
display(sampling_pair[['direction','sampling']+change_fields])
sampling_truth_target=sampling_long[sampling_long.energy_min_ev==ENERGY_MIN_EVQ].copy()
for name in ['density','velocity','temperature','pressure']:
    sampling_truth_target[name+'_error_pct']=100*(sampling_truth_target[name+'_ratio']-1)
sampling_truth_target.to_csv(OUTPUT_DIR/f'sampling_{OUTPUT_TAG}_errors_vs_truth.csv',index=False)
print(f'{ENERGY_MIN_EVQ:g} eV/q cutoff: centre/bin-average errors relative to full-distribution truth')
display(sampling_truth_target[['direction','sampling']+error_fields])
validation.clear_synthetic()


## 解釈の範囲

- 主結果の誤差は指定下限の出力と全分布の真値との差であり、30 eV/qとの出力差ではない。
- 補足の「参照下限と指定下限で近い」と言えるのは、ここで指定した分布・流速・方向・応答モデルに対してである。実観測での下限依存性はまだ検証していない。
- 未観測の冷たい成分を持つ二成分分布などでは、観測されたpartial温度が1 keVでも、この単一Maxwell分布の結果をそのまま適用できない。
- 実関数と独立離散計算の一致は、与えられたビン入力に対する処理の整合性を検証する。全速度空間のmomentとの一致を保証するものではない。
- 速度比は入力ドリフト方向の射影に関する値。研究で使う衛星速度補正後の成分、波数換算やフィット結果への伝播はこのノートでは計算していない。
- 元の30 eVノートは解像度収束・無効値処理も含む基礎検証として保持する。この比較は積分下限を変えた際の差を追加するものになる。
